# 資料處理 part

In [85]:
%reset -f

### 路徑設定
- **模組**沿用 `rebuild_data/`（schema / parse）
- **資料根目錄**獨立於 `rebuild.ipynb` 的 `rebuild_root`
- zip **不解壓落地**：`read_tej_zip` 只讀進記憶體，減少 SSD 寫入
- 本 notebook 目前只做**載入**，處理流程後續再寫

In [86]:
import sys
import os
import zipfile
import importlib

import polars as pl

# Python 模組：rebuild_data（與全量重建共用）
modules_root = "/home/newt/quant_project/rebuild_data"
if modules_root not in sys.path:
    sys.path.insert(0, modules_root)

# 每日增量資料根目錄（獨立，勿與 rebuild_root 混用）
update_root = "/mnt/c/Users/Ultra7/TEJ_data/TEJ__data_daily"
zip_dir = os.path.join(update_root, "daily_add")     # TEJ 下載的 zip（直接讀，不解壓）
save_path = os.path.join(update_root, "daily_save")  # 輸出（後續用）

os.makedirs(save_path, exist_ok=True)
print("modules_root:", modules_root)
print("update_root :", update_root)
print("zip_dir     :", zip_dir)
print("save_path   :", save_path)

modules_root: /home/newt/quant_project/rebuild_data
update_root : /mnt/c/Users/Ultra7/TEJ_data/TEJ__data_daily
zip_dir     : /mnt/c/Users/Ultra7/TEJ_data/TEJ__data_daily/daily_add
save_path   : /mnt/c/Users/Ultra7/TEJ_data/TEJ__data_daily/daily_save


In [87]:
from parse import read_tej_txt, read_tej_zip, find_tej_zip


def reload_modules(module_list):
    for mod in module_list:
        importlib.reload(mod)


def extract_zips_inplace(zip_dir: str) -> list[str]:
    """備援：把 zip 內 .txt 解到 zip 同目錄（不另開 daily_work）。

    日常請用 read_tej_zip（記憶體）；只有外部工具一定要實體 txt 時才呼叫。
    """
    stems = []
    for name in sorted(os.listdir(zip_dir)):
        if not name.endswith(".zip"):
            continue
        zpath = os.path.join(zip_dir, name)
        with zipfile.ZipFile(zpath, "r") as zf:
            zf.extractall(zip_dir)
        stems.append(name[:-4])
        print(f"extracted inplace: {name} → {zip_dir}")
    return stems


## 資料載入（僅讀取，不處理）
一律 `read_tej_zip(zip_dir, 名稱前綴, …)`，不解壓到 SSD。
`calendar` / `company_info` 若每日 zip 沒有則 skip。

### OHLCV（Daily_Unadjusted_Price）— 日收盤價（未復權）


In [88]:
import OHLCV
from OHLCV import OHLCV_SCHEMA, encode_ohlcv_codes

OHLCV_df = read_tej_zip(
    zip_dir,
    "未調整股價(日)",
    OHLCV_SCHEMA,
).select(OHLCV.keep_col)

OHLCV_df = OHLCV_df.filter(OHLCV_df["market"].is_in(["TSE", "OTC"]))
OHLCV_df = encode_ohlcv_codes(OHLCV_df)

# 全額交割 → untradable=1；其餘 0（halt 稍後覆寫）
OHLCV_df = OHLCV_df.with_columns(
    [
        (pl.col("full_delivery") == 1)
        .fill_null(False)
        .cast(pl.Int8)
        .alias("untradable"),
        pl.lit(1.0).alias("adj_factor"),
        pl.lit(0).cast(pl.Int8).alias("avoid_early"),
    ]
)

# 檢查 symbol 欄位是否都為 1100~9999 的字串
invalid_symbols = OHLCV_df.filter(
    ~(pl.col("symbol").cast(pl.Int32).is_between(1100, 9999))
)
if invalid_symbols.height > 0:
    print("以下 symbol 不在 1100~9999 範圍：")
    display(invalid_symbols.select("symbol").unique())
else:
    print("所有 symbol 都在 1100~9999 範圍內。")
print("OHLCV_df:", OHLCV_df.shape)
OHLCV_df.head()

所有 symbol 都在 1100~9999 範圍內。
OHLCV_df: (1944, 25)


symbol,date,open,high,low,close,volume,amount,return_day,turnover,market_value_million,log_return_day,numtrades,limit_status,spread,next_open_ref,next_limit_up,next_limit_down,attention_stock,disposed_stock,full_delivery,market,untradable,adj_factor,avoid_early
str,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,f64,f64,f64,f64,i8,i8,i8,i8,i8,f64,i8
"""1101""",2026-08-04,23.6,23.7,23.45,23.55,25392.0,597866.0,-1.0504,0.3375,177171.0,-1.056,9702.0,0,1.0504,23.55,25.9,21.2,0,0,0,1,0,1.0,0
"""1102""",2026-08-04,32.35,32.5,32.2,32.4,10095.0,326168.0,-0.9174,0.2846,114909.0,-0.9217,4600.0,0,0.9174,32.4,35.6,29.2,0,0,0,1,0,1.0,0
"""1103""",2026-08-04,13.0,13.05,12.9,12.95,566.0,7362.0,-1.5209,0.068,10786.0,-1.5326,420.0,0,1.1407,12.95,14.2,11.7,0,0,0,1,0,1.0,0
"""1104""",2026-08-04,26.6,26.6,26.35,26.45,988.0,26166.0,-1.6729,0.1439,18163.0,-1.687,852.0,0,0.9294,26.45,29.05,23.85,0,0,0,1,0,1.0,0
"""1108""",2026-08-04,11.9,12.05,11.85,12.05,243.0,2907.0,-0.4132,0.06,4877.0,-0.4141,123.0,0,1.6529,12.05,13.25,10.85,0,0,0,1,0,1.0,0


In [89]:
unique_symbol = OHLCV_df["symbol"].unique().to_list()
unique_date = OHLCV_df["date"].unique().to_list()
if len(unique_date) != 1:
    raise ValueError("unique_date 不只一個")
else :
    unique_date = unique_date[0]
print(f'unique_symbol: {len(unique_symbol)}')
print(OHLCV_df['market'].unique())
print(OHLCV_df['date'].unique())
#print('6867' in unique_symbol)
#print(OHLCV_df.filter(pl.col("symbol") == '3152').select('untradable'))


unique_symbol: 1944
shape: (2,)
Series: 'market' [i8]
[
	0
	1
]
shape: (1,)
Series: 'date' [date]
[
	2026-08-04
]


### Trading_Halt — 暫停交易


In [90]:
import Trading_Halt
reload_modules([Trading_Halt])
halt_calendar_path = "/home/newt/quant_project/src/manage_database/TW_stock/halt_calendar.json"

trading_halt_df = read_tej_zip(
    zip_dir,
    "暫停交易記錄",
    Trading_Halt.TRADING_HALT_SCHEMA,
).select(Trading_Halt.keep_col)


trading_halt_df = trading_halt_df.filter(
    pl.col("symbol").is_in(unique_symbol)
)
print("trading_halt_df:", trading_halt_df.shape)

if trading_halt_df.height == 0:
    print("當日無暫停交易紀錄，halt_calendar 不更新")
else:
    n_new = Trading_Halt.update_halt_calendar(trading_halt_df, halt_calendar_path)
    print(f"已寫入 halt_calendar：新增 {n_new} 筆 → {halt_calendar_path}")
    display(trading_halt_df)


trading_halt_df: (0, 4)
當日無暫停交易紀錄，halt_calendar 不更新


### 套用 halt_calendar
- date ∈ `[start_halt, resume_trading_date)` → `untradable=1`
- reason 含「減資」→ `avoid_early=T`，標在 **start_halt 前、本批 OHLCV 最後一個交易日** 的 `avoid_early=1`
- calendar 裡有、但本批 OHLCV 沒有的 symbol 會印出提示（例如停牌當天已無報價）
- 套用後：若 `resume_trading_date <= OHLCV 最大日`，從 `halt_calendar.json` 移除該筆


In [91]:
import Trading_Halt

reload_modules([Trading_Halt])
halt_calendar_path = "/home/newt/quant_project/src/manage_database/TW_stock/halt_calendar.json"

OHLCV_df = Trading_Halt.apply_halt_calendar_to_ohlcv(OHLCV_df, halt_calendar_path)
print(
    "after halt_calendar: untradable=1 →",
    OHLCV_df.filter(pl.col("untradable") == 1).height,
    "| avoid_early=1 →",
    OHLCV_df.filter(pl.col("avoid_early") == 1).height,
)
display(OHLCV_df.filter((pl.col("untradable") == 1) | (pl.col("avoid_early") == 1)).head())

n_pruned = Trading_Halt.prune_halt_calendar(OHLCV_df, halt_calendar_path)
print(f"prune_halt_calendar: removed {n_pruned}")

after halt_calendar: untradable=1 → 31 | avoid_early=1 → 0


symbol,date,open,high,low,close,volume,amount,return_day,turnover,market_value_million,log_return_day,numtrades,limit_status,spread,next_open_ref,next_limit_up,next_limit_down,attention_stock,disposed_stock,full_delivery,market,untradable,adj_factor,avoid_early
str,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,f64,f64,f64,f64,i8,i8,i8,i8,i8,f64,i8
"""1213""",2026-08-04,7.45,7.53,7.33,7.5,9.0,67.0,3.5912,0.0159,425.0,3.5282,8.0,0,2.7624,7.5,8.25,6.75,0,0,1,1,1,1.0,0
"""1538""",2026-08-04,10.2,10.2,10.2,10.2,0.0,0.0,0.0,0.0,329.0,0.0,1.0,0,0.0,10.2,11.2,9.18,0,0,1,1,1,1.0,0
"""1589""",2026-08-04,5.54,5.54,5.54,5.54,0.0,0.0,0.0,0.0,848.0,0.0,0.0,0,0.0,5.54,null,null,0,0,0,1,1,1.0,0
"""2314""",2026-08-04,12.0,12.1,12.0,12.05,86.0,1040.0,0.4167,0.031,3346.0,0.4158,103.0,0,0.8333,12.05,13.25,10.85,0,0,1,1,1,1.0,0
"""2321""",2026-08-04,14.55,16.1,14.55,15.0,14.0,222.0,2.0408,0.0462,454.0,2.0203,33.0,0,10.5442,15.0,16.5,13.5,0,0,1,1,1,1.0,0


prune_halt_calendar: removed 0


### Cash_Dividends — 現金股息
對齊 OHLCV `(symbol, date)`、同日去重；保留 `ex_div_ref_price` 供合併時與權利重疊檢查。


In [92]:
import Cash_Dividends
reload_modules([Cash_Dividends])

cash_dividend_df = Cash_Dividends.prepare_for_merge(
    Cash_Dividends.process_cash_dividends(
        read_tej_zip(
            zip_dir,
            "現金股息",
            Cash_Dividends.CASH_DIVIDENDS_SCHEMA,
        )
    ),
    OHLCV_df,
)

# 僅保留有重疊 symbol
cash_dividend_df = cash_dividend_df.filter(
    pl.col("symbol").is_in(unique_symbol)
)

# date 必須等於本批 OHLCV 日；不符 → 下載錯檔，raise 通知（不刪）
wrong_dates = cash_dividend_df.filter(pl.col("date") != unique_date)
if wrong_dates.height > 0:
    print(
        f"⚠ cash_dividend date 與 OHLCV unique_date={unique_date} 不符："
        f"{wrong_dates.height} 筆（可能下載錯日）"
    )
    display(wrong_dates)
    raise ValueError(
        f"cash_dividend 有 {wrong_dates.height} 筆 date != {unique_date}，請檢查下載資料"
    )

# 檢查是否有 prev_close / ex_div_ref_price 為空值（加 assert, 必無空值）
assert cash_dividend_df["ex_div_ref_price"].null_count() == 0, "ex_div_ref_price 有空值"
if "prev_close" in cash_dividend_df.columns:
    assert cash_dividend_df["prev_close"].null_count() == 0, "prev_close 有空值"

has_cash_dividend = cash_dividend_df.height > 0
print("has_cash_dividend:", has_cash_dividend, "| shape:", cash_dividend_df.shape)
if has_cash_dividend:
    display(cash_dividend_df.head())


prev_close / ex_div_ref_price：無空值
cash dividend 無 OHLCV pair：drop 2 筆
has_cash_dividend: True | shape: (22, 4)


symbol,date,adj_factor,ex_div_ref_price
str,date,f64,f64
"""1474""",2026-08-04,1.025,10.0
"""1736""",2026-08-04,1.035857,125.5
"""2348""",2026-08-04,1.125738,66.09
"""2351""",2026-08-04,1.006625,155.47
"""2393""",2026-08-04,1.078947,57.0


### Corporate_Actions — 資本形成（除權／增減資等）


In [93]:
import Corporate_Actions
from Corporate_Actions import CORPORATE_ACTIONS_SCHEMA

ex_rights_df = read_tej_zip(
    zip_dir,
    "資本形成",
    CORPORATE_ACTIONS_SCHEMA,
)

ex_rights_df = ex_rights_df.filter(
    pl.col("symbol").is_in(unique_symbol)
)

print("ex_rights_df:", ex_rights_df.shape)
ex_rights_df.head()

ex_rights_df: (5, 74)


symbol,date,cash_capital_increase_thousand,retained_earnings_increase_thousand,capital_reserve_increase_thousand,capital_reduction_thousand,capital_reduction_cash_per_share,capital_reduction_rate,par_value_change_shares_thousand,par_value_change_ratio,cash_subscription_rate,cash_subscription_price,retained_earnings_stock_rate,capital_reserve_stock_rate,ex_rights_ref_price,prev_close,stock_dividend_yn,private_placement_yn,total_capital_thousand,par_value,currency,total_shares_thousand,distributable_shares_thousand,employee_bonus_shares_thousand,cb_conversion_shares_thousand,pref_conversion_shares_thousand,share_swap_thousand,treasury_cancel_thousand,rsa_cancel_thousand,merger_shares_thousand,transfer_shares_thousand,eso_shares_thousand,rsa_shares_thousand,other_shares_thousand,employee_stock_rate,cb_conversion_price,employee_subscription_rate,public_underwriting_rate,book_building_rate,gdr_rate,specific_rate,capital_change_announce_date,shareholder_meeting_date,sfc_approval_date,book_closure_start,book_closure_end,shareholder_payment_start,shareholder_payment_end,cash_call_start,cash_call_end,payment_certificate_announce_date,payment_certificate_issue_date,payment_certificate_list_date,cash_increase_stock_announce_date,cash_increase_stock_issue_date,cash_increase_stock_list_date,rights_certificate_announce_date,rights_certificate_issue_date,rights_certificate_list_date,employee_bonus_announce_date,employee_bonus_issue_date,employee_bonus_list_date,stock_dividend_announce_date,stock_dividend_issue_date,stock_dividend_list_date,last_transfer_date,margin_buy_suspend_start,margin_suspend_end,margin_sell_suspend_start,short_cover_deadline,cash_increase_failed,capital_increase_fail_reason,other_note,capital_reduction_refund_date
str,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,date,str,str,str,date
"""1815""",2026-08-04,0.0,0.0,0.0,0.0,0.0,null,0.0,null,null,0.0,null,null,null,null,"""N""","""N""",5.854595e6,10.0,"""NTD""",585459.0,585479.0,0.0,0.0,0.0,0.0,0.0,20.0,0.0,0.0,0.0,0.0,0.0,null,0.0,null,null,null,null,null,2026-08-04,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""N""",null,"""限制員工權利新股註銷""",null
"""2348""",2026-08-04,0.0,7587.0,0.0,0.0,0.0,0.0,0.0,null,0.0,0.0,5.0,0.0,66.09,74.4,"""Y""","""N""",1.594856e6,10.0,"""NTD""",159486.0,151735.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-07-17,2026-06-17,2026-07-03,2026-08-06,2026-08-10,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2026-08-05,null,2026-08-03,2026-07-29,2026-07-29,"""N""",null,null,null
"""2812""",2026-08-04,0.0,403449.0,0.0,0.0,0.0,0.0,0.0,null,0.0,0.0,6.7,0.0,18.98,20.65,"""Y""","""N""",6.4250749e7,10.0,"""NTD""",6.425075e6,6.021626e6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-07-16,2026-05-27,null,2026-08-06,2026-08-10,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2026-08-05,null,2026-08-03,2026-07-29,2026-07-29,"""N""",null,null,null
"""2834""",2026-08-04,0.0,680264.0,0.0,0.0,0.0,0.0,0.0,null,0.0,0.0,7.0,0.0,16.72,18.2,"""Y""","""N""",1.03983262e8,10.0,"""NTD""",1.0398326e7,9.718062e6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-07-15,2026-06-18,2026-07-14,2026-08-06,2026-08-10,null,null,null,null,null,null,null,null,null,null,2026-07-20,2026-09-02,2026-09-02,null,null,null,null,null,null,2026-08-05,null,2026-08-03,2026-07-29,2026-07-29,"""N""",null,null,null
"""6469""",2026-08-04,0.0,9000.0,0.0,0.0,0.0,0.0,0.0,null,0.0,0.0,5.995515,0.0,70.04,77.4,"""Y""","""N""",1.591043e6,10.0,"""NTD""",159104.0,150105.0,0.0,0.0,0.0,0.0

#### 配股（盈餘／公積增資）
無配股時 `has_stock_dividend=False`，後續合併可跳過。


In [94]:
reload_modules([Corporate_Actions])
ex_rights_adj_factor_df, has_stock_dividend = Corporate_Actions.process_stock_dividend(ex_rights_df)
print("has_stock_dividend:", has_stock_dividend)
if has_stock_dividend:
    print("ex_rights_adj_factor_df:", ex_rights_adj_factor_df.shape)
    display(ex_rights_adj_factor_df.head())
else:
    print("無配股，跳過後續配股合併")

配股 adj_factor 唯一 OK，筆數: 4
has_stock_dividend: True
ex_rights_adj_factor_df: (4, 4)


symbol,date,adj_factor,ref_price
str,date,f64,f64
"""2348""",2026-08-04,1.125738,66.09
"""2812""",2026-08-04,1.087987,18.98
"""2834""",2026-08-04,1.088517,16.72
"""6469""",2026-08-04,1.105083,70.04


#### 減資
停牌對照用 `halt_calendar.json`（累積）。無減資時 `has_capital_reduction=False`，後續合併可跳過。
有恢復日的減資因子寫入 `capital_reduction.json`；OHLCV 日 ≥ 恢復日則從 JSON 刪除（當日仍用記憶體列合併）。


In [95]:
reload_modules([Corporate_Actions, Trading_Halt])
# 停牌對照用累積 halt_calendar，勿只用當日 zip
halt_for_adj = Trading_Halt.load_halt_calendar(halt_calendar_path)
print("halt_for_adj (calendar):", halt_for_adj.shape)

capital_reduction_path = (
    "/home/newt/quant_project/src/manage_database/TW_stock/capital_reduction.json"
)

capital_reduction_only_df, capital_reduction_halt_df, has_capital_reduction = (
    Corporate_Actions.process_capital_reduction(
        ex_rights_df,
        halt_for_adj,
        exclude_pairs=ex_rights_adj_factor_df.select(["symbol", "date"]),
    )
)

# 有恢復日的減資：寫入 JSON；再併入 pending（含尚未到期的歷史列）
n_cr_new = Corporate_Actions.update_capital_reduction_calendar(
    capital_reduction_only_df, capital_reduction_path
)
print(f"capital_reduction.json 新增 {n_cr_new} 筆 → {capital_reduction_path}")
pending_cr = Corporate_Actions.load_capital_reduction_calendar(capital_reduction_path)
if pending_cr.height:
    capital_reduction_only_df = (
        pl.concat([capital_reduction_only_df, pending_cr], how="diagonal_relaxed")
        .unique(subset=["symbol", "date"], keep="first")
        .sort(["symbol", "date"])
    )

# 合併只用本批 OHLCV 對得到的 date；未來恢復日留在 JSON
_ohlcv_dates = OHLCV_df["date"].unique().to_list()
capital_reduction_only_df = capital_reduction_only_df.filter(
    pl.col("date").is_in(_ohlcv_dates)
)
has_capital_reduction = capital_reduction_only_df.height > 0

# 恢復當天（及之前）從 JSON 刪掉；合併仍用上面記憶體列
ref_date = OHLCV_df["date"].max()
n_cr_pruned = Corporate_Actions.prune_capital_reduction_calendar(
    ref_date, capital_reduction_path
)
print(f"prune_capital_reduction: removed {n_cr_pruned} (ref={ref_date})")

print("has_capital_reduction:", has_capital_reduction)
if has_capital_reduction:
    print("capital_reduction_only_df:", capital_reduction_only_df.shape)
    print("capital_reduction_halt_df:", capital_reduction_halt_df.shape)
    display(capital_reduction_only_df.head())
    display(capital_reduction_halt_df.head())
else:
    print("無減資，跳過後續減資合併")


halt_for_adj (calendar): (3, 4)
capital_reduction_thousand 有值 (非空且大於0) 的資料筆數: 0
其中 ex_rights_ref_price 和 prev_close 皆不為空的筆數: 0
無減資資料，跳過後續減資流程
capital_reduction.json 新增 0 筆 → /home/newt/quant_project/src/manage_database/TW_stock/capital_reduction.json
prune_capital_reduction: removed 0 (ref=2026-08-04)
has_capital_reduction: False
無減資，跳過後續減資合併


#### 面額變更
停牌對照用 `halt_calendar.json`（累積）。無面額變更時 `has_par_value_change=False`，後續合併可跳過。
有恢復日的面額變更因子寫入 `par_value_change.json`；OHLCV 日 ≥ 恢復日則從 JSON 刪除（當日仍用記憶體列合併）。


In [96]:
reload_modules([Corporate_Actions, Trading_Halt])
# 停牌對照用累積 halt_calendar，勿只用當日 zip
halt_for_adj = Trading_Halt.load_halt_calendar(halt_calendar_path)
print("halt_for_adj (calendar):", halt_for_adj.shape)

par_value_change_path = (
    "/home/newt/quant_project/src/manage_database/TW_stock/par_value_change.json"
)

par_value_change_only_df, par_value_change_halt_df, has_par_value_change = (
    Corporate_Actions.process_par_value_change(
        ex_rights_df,
        halt_for_adj,
        exclude_pairs=ex_rights_adj_factor_df.select(["symbol", "date"]),
    )
)

# 有恢復日的面額變更：寫入 JSON；再併入 pending（含尚未到期的歷史列）
n_par_new = Corporate_Actions.update_par_value_change_calendar(
    par_value_change_only_df, par_value_change_path
)
print(f"par_value_change.json 新增 {n_par_new} 筆 → {par_value_change_path}")
pending_par = Corporate_Actions.load_par_value_change_calendar(par_value_change_path)
if pending_par.height:
    par_value_change_only_df = (
        pl.concat([par_value_change_only_df, pending_par], how="diagonal_relaxed")
        .unique(subset=["symbol", "date"], keep="first")
        .sort(["symbol", "date"])
    )

# 合併只用本批 OHLCV 對得到的 date；未來恢復日留在 JSON
_ohlcv_dates = OHLCV_df["date"].unique().to_list()
par_value_change_only_df = par_value_change_only_df.filter(
    pl.col("date").is_in(_ohlcv_dates)
)
has_par_value_change = par_value_change_only_df.height > 0

# 恢復當天（及之前）從 JSON 刪掉；合併仍用上面記憶體列
ref_date = OHLCV_df["date"].max()
n_par_pruned = Corporate_Actions.prune_par_value_change_calendar(
    ref_date, par_value_change_path
)
print(f"prune_par_value_change: removed {n_par_pruned} (ref={ref_date})")

print("has_par_value_change:", has_par_value_change)
if has_par_value_change:
    print("par_value_change_only_df:", par_value_change_only_df.shape)
    print("par_value_change_halt_df:", par_value_change_halt_df.shape)
    display(par_value_change_only_df.head())
    display(par_value_change_halt_df.head())
else:
    print("無面額變更，跳過後續面額變更合併")




halt_for_adj (calendar): (3, 4)
無面額變更資料，跳過後續面額變更流程
par_value_change.json 新增 0 筆 → /home/newt/quant_project/src/manage_database/TW_stock/par_value_change.json
prune_par_value_change: removed 0 (ref=2026-08-04)
has_par_value_change: False
無面額變更，跳過後續面額變更合併


### institutional_return — 籌碼動向＆報酬


In [97]:
import institutional_return

reload_modules([institutional_return])
institutional_return_df = institutional_return.process_institutional_return(
    read_tej_zip(
        zip_dir,
        "籌碼動向＆報酬(2008年起)",
        institutional_return.INSTITUTIONAL_RETURN_SCHEMA,
    )
)
print("institutional_return_df:", institutional_return_df.shape)
institutional_return_df = institutional_return_df.filter(
    pl.col("symbol").is_in(unique_symbol)
)
print("after symbol filter:", institutional_return_df.shape)
institutional_return_df.head()

institutional_return_df: (2333, 31)
after symbol filter: (1944, 31)


symbol,date,foreign_net_buy_lot,it_net_buy_lot,inst_net_buy_lot,margin_net_buy_lot,cash_net_buy_lot,foreign_net_buy_days,it_net_buy_days,dealer_net_buy_days,inst_net_buy_days,margin_net_buy_days,cash_net_buy_days,foreign_streak_return,it_streak_return,dealer_streak_return,inst_streak_return,margin_streak_return,cash_streak_return,foreign_streak_cum_lot,it_streak_cum_lot,dealer_streak_cum_lot,inst_streak_cum_lot,margin_streak_cum_lot,cash_streak_cum_lot,foreign_streak_cum_amt_thousand,it_streak_cum_amt_thousand,dealer_streak_cum_amt_thousand,inst_streak_cum_amt_thousand,margin_streak_cum_amt_thousand,cash_streak_cum_amt_thousand
str,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""1101""",2026-08-04,-12259.0,0.0,-12763.0,0.0,12763.0,-2.0,0.0,-2.0,-2.0,0.0,2.0,-3.0864,-1.0504,-3.0864,-3.0864,-1.0504,-3.0864,-35813.0,0.0,-717.0,-36526.0,0.0,34280.0,-852047.0,0.0,-16962.0,-868913.0,0.0,815190.0
"""1102""",2026-08-04,-3217.0,-1915.0,-5342.0,0.0,5342.0,-2.0,-2.0,-2.0,-2.0,0.0,2.0,-1.8182,-1.8182,-1.8182,-1.8182,-0.9174,-1.8182,-3251.0,-3930.0,-385.0,-7566.0,0.0,7346.0,-105047.0,-127449.0,-12480.0,-244977.0,0.0,237817.0
"""1103""",2026-08-04,-352.0,0.0,-416.0,0.0,416.0,-3.0,0.0,-2.0,-3.0,0.0,3.0,-1.5209,-1.5209,-1.145,-1.5209,-1.5209,-1.5209,-502.0,0.0,-94.0,-582.0,0.0,554.0,-6550.0,0.0,-1226.0,-7591.0,0.0,7221.0
"""1104""",2026-08-04,-774.0,0.0,-806.0,0.0,806.0,-3.0,0.0,-2.0,-3.0,0.0,3.0,-2.7574,-1.6729,-1.306,-2.7574,-1.6729,-2.7574,-983.0,0.0,-71.0,-1037.0,0.0,1024.0,-26139.0,0.0,-1895.0,-27574.0,0.0,27223.0
"""1108""",2026-08-04,-33.0,0.0,-76.0,0.0,76.0,-1.0,0.0,-2.0,-1.0,0.0,1.0,-0.4132,-0.4132,0.8368,-0.4132,-0.4132,-0.4132,-33.0,0.0,-58.0,-76.0,0.0,76.0,-395.0,0.0,-695.0,-909.0,0.0,909.0


### institutional_net_flow — 三大法人買賣超


In [98]:
import institutional_net_flow

reload_modules([institutional_net_flow])
institutional_net_flow_df = institutional_net_flow.process_institutional_net_flow(
    read_tej_zip(
        zip_dir,
        "三大法人買賣超",
        institutional_net_flow.INSTITUTIONAL_NET_FLOW_SCHEMA,
    )
)
print("institutional_net_flow_df:", institutional_net_flow_df.shape)

institutional_net_flow_df = institutional_net_flow_df.filter(
    pl.col("symbol").is_in(unique_symbol)
)
print("after symbol filter:", institutional_net_flow_df.shape)


institutional_net_flow_df.head()

institutional_net_flow_df: (2333, 10)
after symbol filter: (1944, 10)


symbol,date,foreign_holding_pct,it_holding_pct,dealer_holding_pct,foreign_turnover_pct,it_turnover_pct,dealer_turnover_pct,dealer_prop_net_buy_thousand_shares,dealer_hedge_net_buy_thousand_shares
str,date,f64,f64,f64,f64,f64,f64,f64,f64
"""1101""",2026-08-04,14.05,0.15,0.06,0.18,0.0,0.01,-248.0,-256.0
"""1102""",2026-08-04,12.54,5.32,0.4,0.12,0.03,0.01,-210.0,0.0
"""1103""",2026-08-04,4.51,0.0,0.01,0.03,0.0,0.0,-64.0,0.0
"""1104""",2026-08-04,14.69,0.02,0.02,0.07,0.0,0.0,-32.0,0.0
"""1108""",2026-08-04,1.3,0.0,0.02,0.02,0.0,0.01,-43.0,0.0


### Credit_Transactions — 融資融券


In [99]:
import Credit_Transactions

reload_modules([Credit_Transactions])
Credit_Transaction_df = Credit_Transactions.process_credit_transactions(
    read_tej_zip(
        zip_dir,
        "融資融券",
        Credit_Transactions.CREDIT_TRANSACTIONS_SCHEMA,
    )
)
print("Credit_Transaction_df:", Credit_Transaction_df.shape)

Credit_Transaction_df = Credit_Transaction_df.filter(
    pl.col("symbol").is_in(unique_symbol)
)
print("after symbol filter:", Credit_Transaction_df.shape)


Credit_Transaction_df.head()

margin: repay_ratio_na=118, balance_lot=15 (12.71%), balance_0=103 (87.29%)
short: repay_ratio_na=818, balance_lot=15 (1.83%), balance_0=803 (98.17%)
Credit_Transaction_df: (1867, 22)
after symbol filter: (1852, 22)


symbol,date,marginable,margin_usage_rate,short_usage_rate,short_to_margin_ratio,margin_change_ratio,short_change_ratio,margin_volume_pct,short_volume_pct,margin_maintenance_ratio,short_maintenance_ratio,account_maintenance_ratio,margin_buy_lot,margin_sell_lot,margin_cash_repay_lot,short_buy_lot,short_sell_lot,short_stock_repay_lot,short_last_cover_date,margin_cash_repay_ratio,short_stock_repay_ratio
str,date,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,date,f64,f64
"""1101""",2026-08-04,1,1.83,0.0,0.07,3.49,-53.7,3.19,0.06,162.11,195.56,162.15,1390.0,228.0,1.0,31.0,2.0,0.0,null,0.000029,0.0
"""1102""",2026-08-04,1,0.2,0.0,0.4,14.21,0.0,1.13,0.0,161.71,191.55,161.9,224.0,5.0,0.0,0.0,0.0,0.0,null,0.0,0.0
"""1103""",2026-08-04,1,1.96,0.0,0.02,0.02,0.0,8.22,0.0,153.6,195.43,153.62,52.0,41.0,10.0,0.0,0.0,0.0,null,0.002449,0.0
"""1104""",2026-08-04,1,0.99,0.0,0.0,0.95,-100.0,0.81,0.3,154.0,null,154.0,16.0,0.0,0.0,6.0,0.0,0.0,null,0.0,null
"""1108""",2026-08-04,1,0.41,0.04,10.53,0.48,0.0,0.41,0.0,146.82,195.52,153.34,2.0,0.0,0.0,0.0,0.0,0.0,null,0.0,0.0


### Day_Trading_Stats — 當日沖銷交易統計（當沖）


In [100]:
import parse
import Day_Trading_Stats

reload_modules([parse, Day_Trading_Stats])
Day_Trading_Stats_df = Day_Trading_Stats.process_day_trading_stats(
    read_tej_zip(
        zip_dir,
        "當日沖銷交易統計資訊",
        Day_Trading_Stats.DAY_TRADING_STATS_SCHEMA,
    )
)
print("Day_Trading_Stats_df:", Day_Trading_Stats_df.shape)

Day_Trading_Stats_df = Day_Trading_Stats_df.filter(
    pl.col("symbol").is_in(unique_symbol)
)
print("after symbol filter:", Day_Trading_Stats_df.shape)

Day_Trading_Stats_df.head()

Day_Trading_Stats_df: (1703, 5)
after symbol filter: (1690, 5)


symbol,date,day_trade_volume,day_trade_volume_mkt_pct,day_trade_margin_short_offset_pct
str,date,f64,f64,f64
"""1101""",2026-08-04,6063.0,23.88,0.01
"""1102""",2026-08-04,2323.0,23.01,0.0
"""1103""",2026-08-04,59.0,10.41,0.0
"""1104""",2026-08-04,71.0,7.18,0.0
"""1108""",2026-08-04,50.0,20.55,0.0


### Securities_Lending — 借券餘額
rebuild 標為有問題暫不用；每日 zip 有檔可載入，但 keep_col 只留 `symbol`/`date`，不併入 final。


In [101]:
import Securities_Lending

reload_modules([Securities_Lending])
Securities_Lending_df = Securities_Lending.process_securities_lending(
    read_tej_zip(
        zip_dir,
        "借券餘額",
        Securities_Lending.SECURITIES_LENDING_SCHEMA,
    )
)
print("Securities_Lending_df:", Securities_Lending_df.shape)


Securities_Lending_df = Securities_Lending_df.filter(
    pl.col("symbol").is_in(unique_symbol)
)
print("after symbol filter:", Securities_Lending_df.shape)



Securities_Lending_df.head()

Securities_Lending_df: (1727, 2)
after symbol filter: (1713, 2)


symbol,date
str,date
"""1101""",2026-08-04
"""1102""",2026-08-04
"""1103""",2026-08-04
"""1104""",2026-08-04
"""1108""",2026-08-04


### share_distribution — 籌碼分佈


In [102]:
import share_distribution

reload_modules([share_distribution])
share_distribution_df = share_distribution.process_share_distribution(
    read_tej_zip(
        zip_dir,
        "籌碼分佈(2008年起)",
        share_distribution.SHARE_DISTRIBUTION_SCHEMA,
    )
)
print("share_distribution_df:", share_distribution_df.shape)


share_distribution_df = share_distribution_df.filter(
    pl.col("symbol").is_in(unique_symbol)
)
print("after symbol filter:", share_distribution_df.shape)



share_distribution_df.head()

share_distribution_df: (2333, 16)
after symbol filter: (1944, 16)


symbol,date,foreign_trade_share_pct,it_trade_share_pct,dealer_trade_share_pct,credit_trade_share_pct,cash_trade_share_pct,free_float_turnover_pct,foreign_holding_ex_dir_pct,director_holding_pct,director_pledge_pct,margin_short_offset_pct,margin_increase_lot,margin_decrease_lot,short_increase_lot,short_decrease_lot
str,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""1101""",2026-08-04,52.88,0.0,1.96,0.0,45.17,0.37,14.05,8.41,0.65,null,0.0,0.0,0.0,0.0
"""1102""",2026-08-04,41.48,9.76,2.54,0.0,46.21,0.39,12.54,26.19,2.14,null,0.0,0.0,0.0,0.0
"""1103""",2026-08-04,39.58,0.0,5.65,0.0,54.77,0.08,4.51,16.51,0.0,null,0.0,0.0,0.0,0.0
"""1104""",2026-08-04,46.04,0.0,1.74,0.0,52.22,0.15,14.69,4.58,0.0,null,0.0,0.0,0.0,0.0
"""1108""",2026-08-04,32.72,0.0,8.85,0.0,58.44,0.06,1.3,3.95,0.0,null,0.0,0.0,0.0,0.0


### Company_Info — 公司基本資料／產業（每日）
只保留 `symbol` / `TSE產業名`→`industry` / `TEJ子產業名`→`subindustry`。
再用 `setting_files/*_rebuild.json` 編成整數；若出現對照表沒有的產業／子產業會印出提示。


In [103]:
import company_info_daily
import Company_Info

reload_modules([company_info_daily, Company_Info])
setting_file = "/home/newt/quant_project/setting_files"

try:
    company_info_df = company_info_daily.process_company_info_daily(
        read_tej_zip(zip_dir, "基本資料", company_info_daily.COMPANY_INFO_DAILY_SCHEMA)
    )
    company_info_df, new_industries, new_subindustries = (
        company_info_daily.encode_daily_company_info(company_info_df, setting_file)
    )
    company_info_df = company_info_df.filter(
        pl.col("symbol").is_in(unique_symbol)
    )
    print("company_info_df:", company_info_df.shape, company_info_df.dtypes)
    if new_industries:
        print(f"⚠ 新 industry（不在對照表，已編成 null）: {new_industries}")
    if new_subindustries:
        print(f"⚠ 新 subindustry（不在對照表，已編成 null）: {new_subindustries}")
    if not new_industries and not new_subindustries:
        print("產業對照：無新增")
    display(company_info_df.head())
except FileNotFoundError:
    company_info_df = None
    print("skip company_info：zip_dir 無「基本資料」開頭的 zip")


company_info_df: (1944, 3) [String, Int16, Int16]
產業對照：無新增


symbol,industry,subindustry
str,i16,i16
"""1101""",0,0
"""1102""",0,0
"""1103""",0,0
"""1104""",0,0
"""1108""",0,0


## 載入摘要 — 各表 shape 一覽


In [104]:
_loaded = {
    "暫停交易": trading_halt_df,
    "日收盤價（未復權）": OHLCV_df,
    "現金股息": cash_dividend_df,
    "配股復權因子": ex_rights_adj_factor_df,
    "減資復權因子": capital_reduction_only_df,
    "減資停牌區間": capital_reduction_halt_df,
    "面額變更復權因子": par_value_change_only_df,
    "面額變更停牌區間": par_value_change_halt_df,
    "資本形成（原始）": ex_rights_df,
    "籌碼動向＆報酬": institutional_return_df,
    "三大法人買賣超": institutional_net_flow_df,
    "融資融券": Credit_Transaction_df,
    "當沖統計": Day_Trading_Stats_df,
    "借券餘額": Securities_Lending_df,
    "籌碼分佈": share_distribution_df,
    "公司基本資料": company_info_df,
}
for name, df in _loaded.items():
    if df is None:
        print(f"{name:20s}  SKIP")
    else:
        print(f"{name:20s}  {df.shape[0]:>8} rows × {df.shape[1]} cols")

print("有配股     :", has_stock_dividend)
print("有減資     :", has_capital_reduction)
print("有面額變更 :", has_par_value_change)

暫停交易                         0 rows × 4 cols
日收盤價（未復權）                 1944 rows × 25 cols
現金股息                        22 rows × 4 cols
配股復權因子                       4 rows × 4 cols
減資復權因子                       0 rows × 7 cols
減資停牌區間                       0 rows × 6 cols
面額變更復權因子                     0 rows × 7 cols
面額變更停牌區間                     0 rows × 6 cols
資本形成（原始）                     5 rows × 74 cols
籌碼動向＆報酬                   1944 rows × 31 cols
三大法人買賣超                   1944 rows × 10 cols
融資融券                      1852 rows × 22 cols
當沖統計                      1690 rows × 5 cols
借券餘額                      1713 rows × 2 cols
籌碼分佈                      1944 rows × 16 cols
公司基本資料                    1944 rows × 3 cols
有配股     : True
有減資     : False
有面額變更 : False


## 合併流程
現金除息先掛；配股／減資／面額與現金同日：參考價相同丟權利側，不同 raise。最後 `fill_margin_net_buy_lot_with_zero`。


In [105]:
import Credit_Transactions
reload_modules([Corporate_Actions, Credit_Transactions])


def _join_mul_adj(left: pl.DataFrame, right: pl.DataFrame, name: str = "adj") -> pl.DataFrame:
    """left join right 的 adj_factor：相乘後只留乘積（右為 null 當 1）。"""
    if right.height and left.join(
        right.select(["symbol", "date"]), on=["symbol", "date"], how="inner"
    ).height == 0:
        print(
            f"⚠ 警告: {name} 與 final 無 (symbol,date) 交集，日期可能有錯"
            f"（right dates={sorted(right['date'].unique().to_list())}）"
        )
    out = left.join(
        right.select(["symbol", "date", "adj_factor"]),
        on=["symbol", "date"],
        how="left",
    )
    return (
        out.with_columns(
            (pl.col("adj_factor") * pl.col("adj_factor_right").fill_null(1.0)).alias(
                "adj_factor"
            )
        ).drop("adj_factor_right")
    )


def _join_checked(
    left: pl.DataFrame,
    right: pl.DataFrame,
    name: str,
    on: list[str] | None = None,
) -> pl.DataFrame:
    """left join；欄位未增加或 key 無交集 → 警告（常見於日期錯）。"""
    on = on or ["symbol", "date"]
    n_before = left.width
    if right.height == 0:
        print(f"⚠ 警告: {name} 為空，合併不會增加欄位")
        return left
    if set(on) <= set(right.columns):
        n_hit = left.select(on).join(right.select(on), on=on, how="inner").height
        if n_hit == 0:
            extra = ""
            if "date" in on:
                extra = f"（right dates={sorted(right['date'].unique().to_list())}）"
            print(f"⚠ 警告: {name} 與 final 無 {on} 交集，日期可能有錯{extra}")
    out = left.join(right, on=on, how="left")
    if out.width <= n_before:
        print(
            f"⚠ 警告: {name} 合併後欄位未增加（{n_before}→{out.width}），"
            "可能只有 symbol/date 或日期有錯"
        )
    return out


final_df = OHLCV_df.clone()

# 現金除息先掛；權利側再與現金做同日參考價重疊檢查（同價丟權利／異價 raise）
cash_for_overlap = cash_dividend_df if has_cash_dividend else None
if has_cash_dividend:
    final_df = _join_mul_adj(final_df, cash_dividend_df, "cash_dividend")

if has_stock_dividend:
    final_df = _join_mul_adj(
        final_df,
        Corporate_Actions.filter_rights_vs_cash_overlap(
            ex_rights_adj_factor_df, cash_for_overlap
        ),
        "stock_dividend",
    )

if has_capital_reduction:
    final_df = _join_mul_adj(
        final_df,
        Corporate_Actions.filter_rights_vs_cash_overlap(
            capital_reduction_only_df, cash_for_overlap
        ),
        "capital_reduction",
    )

if has_par_value_change:
    final_df = _join_mul_adj(
        final_df,
        Corporate_Actions.filter_rights_vs_cash_overlap(
            par_value_change_only_df, cash_for_overlap
        ),
        "par_value_change",
    )

final_df = _join_checked(final_df, institutional_return_df, "institutional_return")
final_df = _join_checked(final_df, institutional_net_flow_df, "institutional_net_flow")
final_df = _join_checked(final_df, Credit_Transaction_df, "Credit_Transaction")
final_df = _join_checked(final_df, Day_Trading_Stats_df, "Day_Trading_Stats")
# Securities_Lending 暫不進 DB（keep 只剩 symbol/date）；join 無新欄，略過
final_df = _join_checked(final_df, share_distribution_df, "share_distribution")
if company_info_df is not None:
    final_df = _join_checked(
        final_df, company_info_df, "company_info", on=["symbol"]
    )

final_df = Credit_Transactions.fill_margin_net_buy_lot_with_zero(final_df)


同日同參考價 4 筆 → 丟掉權利側、保留現金除息
shape: (4, 6)
┌────────┬────────────┬───────────┬──────────┬────────────┬──────────┐
│ symbol ┆ date       ┆ ref_price ┆ cash_ref ┆ adj_factor ┆ cash_adj │
│ ---    ┆ ---        ┆ ---       ┆ ---      ┆ ---        ┆ ---      │
│ str    ┆ date       ┆ f64       ┆ f64      ┆ f64        ┆ f64      │
╞════════╪════════════╪═══════════╪══════════╪════════════╪══════════╡
│ 2348   ┆ 2026-08-04 ┆ 66.09     ┆ 66.09    ┆ 1.125738   ┆ 1.125738 │
│ 2812   ┆ 2026-08-04 ┆ 18.98     ┆ 18.98    ┆ 1.087987   ┆ 1.087987 │
│ 2834   ┆ 2026-08-04 ┆ 16.72     ┆ 16.72    ┆ 1.088517   ┆ 1.088517 │
│ 6469   ┆ 2026-08-04 ┆ 70.04     ┆ 70.04    ┆ 1.105083   ┆ 1.105083 │
└────────┴────────────┴───────────┴──────────┴────────────┴──────────┘


In [106]:
import duckdb

duckdb_path = "/home/newt/quant_project/duckdb/daily_data.duckdb"
table_name = "daily_data"

conn = duckdb.connect(duckdb_path, read_only=True)
_desc = conn.execute(f"DESCRIBE {table_name}").fetchall()

# --- 日期檢查：final_df 的 date 是否已在 DB ---
final_dates = sorted(final_df["date"].unique().to_list())
db_max_date = conn.execute(f"SELECT MAX(date) FROM {table_name}").fetchone()[0]
existing_dates = {
    r[0]
    for r in conn.execute(
        f"SELECT DISTINCT date FROM {table_name} WHERE date IN (SELECT UNNEST(?::DATE[]))",
        [final_dates],
    ).fetchall()
}
already_in_db = [d for d in final_dates if d in existing_dates]
new_dates = [d for d in final_dates if d not in existing_dates]

print(f"final_df dates ({len(final_dates)}): {final_dates}")
print(f"DB MAX(date): {db_max_date}")
print(f"已存在於 DB: {already_in_db if already_in_db else '無'}")
print(f"DB 尚無（可寫入）: {new_dates if new_dates else '無'}")
dates_ok = len(already_in_db) == 0
print(f"dates_ok (final 日期皆尚未在 DB): {dates_ok}")
if not dates_ok:
    print("→ 這些日期已在 DB，寫入前請確認是否要覆蓋／跳過")

conn.close()

db_cols = [r[0] for r in _desc]
db_types = {r[0]: r[1] for r in _desc}
final_cols = list(final_df.columns)

only_in_db = [c for c in db_cols if c not in set(final_cols)]      # final 缺
only_in_final = [c for c in final_cols if c not in set(db_cols)]  # final 多
common = [c for c in db_cols if c in set(final_cols)]

print(f"\nDB {table_name}: {len(db_cols)} cols | final_df: {len(final_cols)} cols | 共同: {len(common)}")
print(f"\nfinal 缺少（DB 有）: {len(only_in_db)}")
for c in only_in_db:
    print(f"  - {c:40s} {db_types[c]}")
print(f"\nfinal 多出（DB 無）: {len(only_in_final)}")
for c in only_in_final:
    print(f"  + {c}")

cols_ok = len(only_in_db) == 0
print(f"\ncols_ok (final 涵蓋全部 DB 欄位): {cols_ok}")
if not cols_ok:
    print("→ 合併前請補齊缺少欄位；多出欄位可之後再決定是否寫入 DB")


final_df dates (1): [datetime.date(2026, 8, 4)]
DB MAX(date): 2026-08-03
已存在於 DB: 無
DB 尚無（可寫入）: [datetime.date(2026, 8, 4)]
dates_ok (final 日期皆尚未在 DB): True

DB daily_data: 101 cols | final_df: 101 cols | 共同: 101

final 缺少（DB 有）: 0

final 多出（DB 無）: 0

cols_ok (final 涵蓋全部 DB 欄位): True


In [107]:
#final_df.filter(pl.col("symbol") == '2380').select(['symbol','date','adj_factor'])

# 寫入 DuckDB
前置：`dates_ok` 且 `cols_ok`。執行後會互動詢問確認，輸入 `y`／`yes` 才寫入。
只寫入 DB 既有欄位（`db_cols` 順序）；final 多出的欄位不寫。


In [108]:
if not (dates_ok and cols_ok):
    print(f"跳過寫入：dates_ok={dates_ok}, cols_ok={cols_ok}")
else:
    print(f"準備寫入 {table_name}：{len(final_df)} 列，dates={final_dates}")
    print(f"DB：{duckdb_path}")
    ans = input("確認寫入？[y/N] ").strip().lower()
    if ans not in ("y", "yes"):
        print("已取消，未寫入")
    else:
        write_df = final_df.select(db_cols)
        con = duckdb.connect(duckdb_path, read_only=False)
        n_before = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
        con.register("_daily_append", write_df)
        con.execute(f"INSERT INTO {table_name} BY NAME SELECT * FROM _daily_append")
        n_after = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
        max_date = con.execute(f"SELECT MAX(date) FROM {table_name}").fetchone()[0]
        con.close()
        print(f"已寫入 {n_after - n_before} 列（dates={final_dates}）")
        print(f"表總列數: {n_before} → {n_after} | MAX(date): {max_date}")


準備寫入 daily_data：1944 列，dates=[datetime.date(2026, 8, 4)]
DB：/home/newt/quant_project/duckdb/daily_data.duckdb
已寫入 1944 列（dates=[datetime.date(2026, 8, 4)]）
表總列數: 4794384 → 4796328 | MAX(date): 2026-08-04


## 歸檔 zip → `save_path`
把 `zip_dir`（`daily_add`）內的 `.zip` 移到 `save_path`。
若有 `final_dates`，放到對應日期子目錄（例如 `daily_save/2026-06-01/`）。執行後互動確認。


In [109]:
import shutil

_zips = sorted(n for n in os.listdir(zip_dir) if n.endswith(".zip"))
if not _zips:
    print(f"zip_dir 無 .zip：{zip_dir}")
else:
    _dest_dir = save_path
    if final_dates:
        _dest_dir = os.path.join(save_path, final_dates[0].isoformat())
    print(f"來源: {zip_dir}")
    print(f"目的: {_dest_dir}")
    for n in _zips:
        print(f"  - {n}")
    ans = input(f"確認移動 {len(_zips)} 個 zip？[y/N] ").strip().lower()
    if ans not in ("y", "yes"):
        print("已取消，未移動")
    else:
        os.makedirs(_dest_dir, exist_ok=True)
        moved, skipped = [], []
        for n in _zips:
            src = os.path.join(zip_dir, n)
            dst = os.path.join(_dest_dir, n)
            if os.path.exists(dst):
                print(f"skip（已存在）: {dst}")
                skipped.append(n)
                continue
            shutil.move(src, dst)
            moved.append(n)
            print(f"moved: {n}")
        print(f"完成：moved={len(moved)}, skipped={len(skipped)}")


來源: /mnt/c/Users/Ultra7/TEJ_data/TEJ__data_daily/daily_add
目的: /mnt/c/Users/Ultra7/TEJ_data/TEJ__data_daily/daily_save/2026-08-04
  - 三大法人買賣超 - 2026-08-04 211718.zip
  - 借券餘額 - 2026-08-04 211821.zip
  - 基本資料 - 2026-08-04 211808.zip
  - 暫停交易記錄 - 2026-08-04 211800.zip
  - 未調整股價(日) - 2026-08-04 211831.zip
  - 現金股息 - 2026-08-04 212153.zip
  - 當日沖銷交易統計資訊 - 2026-08-04 211853.zip
  - 籌碼分佈(2008年起) - 2026-08-04 211914.zip
  - 籌碼動向＆報酬(2008年起) - 2026-08-04 211729.zip
  - 融資融券 - 2026-08-04 211750.zip
  - 資本形成 - 2026-08-04 224214.zip
moved: 三大法人買賣超 - 2026-08-04 211718.zip
moved: 借券餘額 - 2026-08-04 211821.zip
moved: 基本資料 - 2026-08-04 211808.zip
moved: 暫停交易記錄 - 2026-08-04 211800.zip
moved: 未調整股價(日) - 2026-08-04 211831.zip
moved: 現金股息 - 2026-08-04 212153.zip
moved: 當日沖銷交易統計資訊 - 2026-08-04 211853.zip
moved: 籌碼分佈(2008年起) - 2026-08-04 211914.zip
moved: 籌碼動向＆報酬(2008年起) - 2026-08-04 211729.zip
moved: 融資融券 - 2026-08-04 211750.zip
moved: 資本形成 - 2026-08-04 224214.zip
完成：moved=11, skipped=0
